# 06 Optimization Techniques

## 📚 Learning Objectives

By completing this notebook (~20 min), you will:
- Compare SGD and Adam optimizers on the same task (MNIST digits)
- See how the choice of optimizer affects loss curves and convergence
- Understand why we often use Adam instead of plain SGD

---

## 🌍 Real life

**Where is this used?** Every time you call `model.compile(optimizer='adam', ...)` or train a neural network, you choose an **optimizer** that uses the gradients from backprop to update the weights.

**In this notebook we use** **optimizers** (SGD and Adam) to **update the weights** so the network learns. We use **Adam** instead of plain **SGD** in many projects **because** Adam adapts the learning rate per parameter and usually **converges faster** and more reliably.

**📌 Covers slide(s):** Part of Unit 1 sequence (training flow). *Do after 05_backpropagation_detailed; then move to Unit 2.*

---

**Before starting:** Run the imports cell below. Requires TensorFlow.


## Theory (short)

- **Optimizer** = algorithm that uses gradients to update weights: `new_weight = old_weight - learning_rate * gradient` (or a variant).
- **SGD (Stochastic Gradient Descent):** uses the gradient directly; simple but can be slow and sensitive to learning rate.
- **Adam:** adapts a learning rate per parameter (momentum + scaling); usually converges faster and is less sensitive to the initial learning rate.
- **Data flow:** forward pass → loss → backprop (gradients) → **optimizer** updates weights → repeat.

### Key math (required)

- **SGD:** \(w_{t+1} = w_t - \eta\, g_t\) where \(g_t = \nabla_w L\) (gradient of loss w.r.t. \(w\)) and \(\eta\) is the learning rate.
- **Adam:** maintains running estimates of the first moment \(m\) and second moment \(v\) of gradients; update: \(m_t = \beta_1 m_{t-1} + (1-\beta_1)g_t\), \(v_t = \beta_2 v_{t-1} + (1-\beta_2)g_t^2\), then bias-correct and apply \(w_{t+1} = w_t - \eta\,\hat{m}_t/(\sqrt{\hat{v}_t}+\epsilon)\). Typical \(\beta_1=0.9\), \(\beta_2=0.999\). See slides or Kingma & Ba (2014) for full form.

**The steps below put this theory into code.**

💡 **If this is unclear:** Focus on the **plot at the end**: one curve is SGD, one is Adam; Adam usually reaches lower loss in the same number of epochs. The main idea: optimizers use gradients to update weights; Adam adapts the step size per parameter. If stuck, tell your instructor: "I didn't get the SGD vs Adam part in notebook 06."


## 📥 Inputs & 📤 Outputs

**Inputs:** MNIST (we use a subset of 10k training samples), TensorFlow/Keras, NumPy, Matplotlib. We build two identical small models in the notebook and train one with SGD and one with Adam.

**Dataset:** Real — MNIST (subset for SGD vs Adam comparison).

**Outputs:** Printed training loss and accuracy per epoch for SGD, then for Adam; then one plot with two curves (SGD train loss vs Adam train loss). Adam typically reaches lower loss in the same number of epochs.


In [ ]:
# Step 1: Imports
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
print(f'PyTorch {torch.__version__}')
print('✅ Imports OK.')

In [ ]:
# Step 2: Load MNIST (small subset so it runs fast)
transform = transforms.ToTensor()
train_data = datasets.MNIST('/tmp/mnist_data', train=True, download=True, transform=transform)
# Use first 5 000 samples for speed
subset = torch.utils.data.Subset(train_data, range(5000))
loader = DataLoader(subset, batch_size=128, shuffle=True)
print('Training on', len(subset), 'samples')

In [ ]:
# Step 3: Factory function — same architecture used for both optimizers
def make_model():
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, 64), nn.ReLU(),
        nn.Linear(64, 10),
    )

model_sgd  = make_model()
model_adam = make_model()
# Copy weights so both models start identically
model_adam.load_state_dict(model_sgd.state_dict())
print('Both models initialised with identical weights')

In [ ]:
# Step 4: Train with SGD
# SGD has a fixed learning rate; it takes longer to converge than Adam on many tasks.
criterion  = nn.CrossEntropyLoss()
opt_sgd    = optim.SGD(model_sgd.parameters(), lr=0.01)
losses_sgd = []
for epoch in range(5):
    model_sgd.train()
    epoch_loss = 0.0
    for xb, yb in loader:
        opt_sgd.zero_grad()
        loss = criterion(model_sgd(xb), yb)
        loss.backward()
        opt_sgd.step()
        epoch_loss += loss.item()
    losses_sgd.append(epoch_loss / len(loader))
    print(f'SGD  epoch {epoch+1}: loss={losses_sgd[-1]:.4f}')

In [ ]:
# Step 5: Train with Adam (same architecture, same data, same starting weights)
# Adam adapts the learning rate per parameter — usually converges faster.
opt_adam    = optim.Adam(model_adam.parameters(), lr=1e-3)
losses_adam = []
for epoch in range(5):
    model_adam.train()
    epoch_loss = 0.0
    for xb, yb in loader:
        opt_adam.zero_grad()
        loss = criterion(model_adam(xb), yb)
        loss.backward()
        opt_adam.step()
        epoch_loss += loss.item()
    losses_adam.append(epoch_loss / len(loader))
    print(f'Adam epoch {epoch+1}: loss={losses_adam[-1]:.4f}')

In [ ]:
# Step 6: Plot loss comparison — SGD vs Adam
plt.figure(figsize=(8, 4))
plt.plot(losses_sgd,  label='SGD  (lr=0.01)', marker='o')
plt.plot(losses_adam, label='Adam (lr=0.001)', marker='s')
plt.title('SGD vs Adam — Training Loss (5 epochs, 5k MNIST samples)')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.legend(); plt.tight_layout(); plt.show()
print('Adam final loss:', round(losses_adam[-1], 4))
print('SGD  final loss:', round(losses_sgd[-1],  4))

## 🌍 Real-World Worked Example — Comparing Optimizers on Image Classification

**Industry context:** Choosing the right optimizer directly impacts training speed and final accuracy.  
OpenAI uses AdamW for GPT models; Google used Adafactor for T5.

We compare SGD, RMSprop, and Adam on the **Digits** dataset to see the difference.

In [ ]:
import torch, torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np, matplotlib.pyplot as plt

digits = load_digits()
X = StandardScaler().fit_transform(digits.data.astype(np.float32))
y = digits.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
Xtr = torch.tensor(X_tr); Ytr = torch.tensor(y_tr, dtype=torch.long)
Xte = torch.tensor(X_te); Yte = torch.tensor(y_te, dtype=torch.long)

def make_model():
    return nn.Sequential(nn.Linear(64,128), nn.ReLU(), nn.Linear(128,10))

optimizers = {
    'SGD (lr=0.01)':   lambda m: torch.optim.SGD(m.parameters(), lr=0.01),
    'RMSprop':         lambda m: torch.optim.RMSprop(m.parameters(), lr=1e-3),
    'Adam':            lambda m: torch.optim.Adam(m.parameters(), lr=1e-3),
}
results = {}
for name, opt_fn in optimizers.items():
    model = make_model()
    opt   = opt_fn(model)
    loss_fn = nn.CrossEntropyLoss()
    curve = []
    for _ in range(150):
        model.train()
        loss = loss_fn(model(Xtr), Ytr)
        opt.zero_grad(); loss.backward(); opt.step()
        curve.append(loss.item())
    model.eval()
    with torch.no_grad():
        acc = (model(Xte).argmax(1)==Yte).float().mean().item()
    results[name] = (curve, acc)
    print(f"{name:20s} → final acc: {acc*100:.1f}%")

plt.figure(figsize=(10,4))
for name,(curve,acc) in results.items():
    plt.plot(curve, label=f"{name} (acc={acc*100:.0f}%)")
plt.title("Optimizer Comparison — Digit Classification (Real-World)"); plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.legend(); plt.tight_layout(); plt.show()

---

## 🧩 Mini-exercise

**Try it (choose one):** Train a third model with **RMSprop** (same architecture, 2 epochs) and add its loss curve to the plot. Compare SGD, Adam, and RMSprop. Or change the learning rate for SGD (e.g. 0.1 or 0.01) and see how the curve changes.

---

## ✅ Summary

**What you did:**
- Loaded MNIST subset and built two identical small models.
- Trained one with **SGD** and one with **Adam** for 2 epochs each.
- Plotted loss curves to compare; Adam often converges faster.

**In real life you'd also:** try different learning rates, use more epochs, and possibly try other optimizers (RMSprop, AdamW).

**The main idea:** The optimizer uses gradients from backprop to update weights; Adam adapts the step size per parameter and is often a good default.

**Next:** Unit 2 CNNs: e.g. `unit2-cnns/examples/01_cnn_architecture.ipynb`. Optionally, `07_image_processing_feature_extraction` (Unit 1) covers image basics before CNNs.


## 📚 References & Further Reading

**Papers:**
- Kingma & Ba (2015) — [Adam: A Method for Stochastic Optimization](https://arxiv.org/abs/1412.6980)
- Loshchilov & Hutter (2019) — [Decoupled Weight Decay Regularization (AdamW)](https://arxiv.org/abs/1711.05101)

**PyTorch Docs:** [torch.optim](https://pytorch.org/docs/stable/optim.html)

**State-of-the-Art:** AdamW + cosine LR schedule is the default for training GPT-4, LLaMA, and Stable Diffusion.